# Part 1 — Student Migration Graph Analysis

To analyse student flows from 10 countries, 206,796 students are considered. For each student, the country of origin (O) and arrival (A) are recorded. The data is stored in `migrations.csv` as a matrix $A$ where $A[a,o]$ contains the number of students moving from country $o$ to country $a$.

We analyse the international student migration dataset using **networkx** (Python equivalent of R's igraph).  
The matrix `migrations.csv` encodes directed flows: **columns = countries of origin (O), rows = countries of arrival (A)**.  
Each cell value is the number of students who migrated from origin country (column) to arrival country (row).

**Q1.** In the literature, how is matrix $A$ referred to? Comment and justify the values of parameters `mode`, `weighted`, and `diag` in the context of building a graph from this matrix.

## Cell 1 — Load data

In [ ]:
import pandas as pd

# Load migrations matrix (sep=';', first column = row names)
A = pd.read_csv('../migrations.csv', sep=';', index_col=0)

# Strip trailing 'O' (Origin) from column names and 'A' (Arrival) from row index
A.columns = [c.rstrip('O') for c in A.columns]
A.index   = [r.rstrip('A') for r in A.index]

print('Shape:', A.shape)
A.head()

## Cell 2 — Build directed weighted graph

We replicate `igraph::graph.adjacency(..., mode="directed", weighted=TRUE, diag=FALSE)`.  
An edge goes **from** origin (column) **to** arrival (row), with `weight = number of students`.

In [ ]:
import networkx as nx

countries = list(A.columns)   # same 10 countries for both axes after stripping

G = nx.DiGraph()
G.add_nodes_from(countries)

for arrival in A.index:
    for origin in A.columns:
        weight = A.loc[arrival, origin]
        if weight > 0:
            G.add_edge(origin, arrival, weight=int(weight))

print(f'Nodes ({G.number_of_nodes()}): {list(G.nodes())}')
print(f'Edges: {G.number_of_edges()}')

**Q2.** Propose and run Python code to plot the graph, using one vertex per country. Use arc widths proportional to their weights, and scale the weights by a 1/3,000 factor.

## Cell 3 — Plot directed graph with edge widths proportional to weight

In [ ]:
import matplotlib.pyplot as plt

pos = nx.spring_layout(G, seed=42)

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
edge_widths  = [w / 3000 for w in edge_weights]

fig, ax = plt.subplots(figsize=(12, 8))

nx.draw_networkx(
    G,
    pos=pos,
    with_labels=True,
    labels={n: n for n in G.nodes()},
    node_color='steelblue',
    node_size=1200,
    font_size=8,
    font_color='white',
    edge_color='gray',
    width=edge_widths,
    arrows=True,
    arrowsize=15,
    ax=ax
)

ax.set_title('Student Migration Network\n(edge width ∝ weight / 3000)', fontsize=13)
plt.tight_layout()
plt.show()

**Q3.** Using visual information from the graph obtained in Q2 only, what are the most striking features of student flows?

**Q4.** Propose a formula for the density of directed graphs, defined as the ratio of the actual number of arcs on the maximal number of arcs (forbidding self-loops). Provide Python commands to compute the graph order, size, density, and diameter (ignoring weights) and provide those values.

## Cell 4 — Graph statistics: order, size, density, diameter

In [ ]:
# Unweighted version for diameter (weights ignored)
G_unweighted = nx.DiGraph()
G_unweighted.add_nodes_from(G.nodes())
G_unweighted.add_edges_from(G.edges())

order   = G.number_of_nodes()
size    = G.number_of_edges()
density = nx.density(G)

# diameter requires strongly connected graph; use weakly connected undirected projection as fallback
if nx.is_strongly_connected(G_unweighted):
    diameter = nx.diameter(G_unweighted)
    conn_type = 'strongly connected'
else:
    G_undirected = G_unweighted.to_undirected()
    diameter = nx.diameter(G_undirected)
    conn_type = 'undirected projection (graph not strongly connected)'

print(f'Order (nodes)  : {order}')
print(f'Size  (edges)  : {size}')
print(f'Density        : {density:.4f}')
print(f'Diameter       : {diameter}  [{conn_type}]')

## Cell 5 — Bipartite graph

We replicate `igraph::graph.incidence()`.  
Origin countries (red) and arrival countries (green) are **separate node sets**, even though they represent the same 10 countries.  
Edges carry the student-flow weight.

In [ ]:
import matplotlib.patches as mpatches

B = nx.Graph()

# Two distinct node sets
origin_nodes  = [f'{c}_origin'  for c in countries]
arrival_nodes = [f'{c}_arrival' for c in countries]

B.add_nodes_from(origin_nodes,  bipartite=0)   # top set
B.add_nodes_from(arrival_nodes, bipartite=1)   # bottom set

for arrival_country in A.index:
    for origin_country in A.columns:
        w = A.loc[arrival_country, origin_country]
        if w > 0:
            B.add_edge(
                f'{origin_country}_origin',
                f'{arrival_country}_arrival',
                weight=int(w)
            )

# Layout: origins on the left, arrivals on the right
pos_B = {}
n = len(countries)
for i, node in enumerate(origin_nodes):
    pos_B[node] = (0, i)
for i, node in enumerate(arrival_nodes):
    pos_B[node] = (2, i)

node_colors = ['red'   if n in origin_nodes  else 'green' for n in B.nodes()]
labels_B    = {n: n.replace('_origin', '').replace('_arrival', '') for n in B.nodes()}
edge_widths_B = [B[u][v]['weight'] / 3000 for u, v in B.edges()]

fig, ax = plt.subplots(figsize=(10, 10))
nx.draw_networkx(
    B, pos=pos_B,
    labels=labels_B,
    node_color=node_colors,
    node_size=1000,
    font_size=7,
    edge_color='gray',
    width=edge_widths_B,
    ax=ax
)
red_patch   = mpatches.Patch(color='red',   label='Origin countries')
green_patch = mpatches.Patch(color='green', label='Arrival countries')
ax.legend(handles=[red_patch, green_patch], fontsize=10)
ax.set_title('Bipartite Student Migration Graph', fontsize=13)
plt.tight_layout()
plt.show()

**Q5.** Display the graph as an undirected, weighted bipartite graph $B$. Perform graph clustering on $B$ and justify your choice of algorithm. Clusters may contain both countries of departure and arrival. Plot the yielded graph partition using different vertex colors for clusters.

## Cell 6 — Community detection

We run the **Louvain** algorithm on the undirected weighted version of the migration graph (equivalent of igraph's community detection on the bipartite graph).  
Each community is given a distinct colour.

In [ ]:
import importlib, sys

# Build undirected weighted graph for community detection
G_undirected_w = G.to_undirected()

# Try Louvain (requires networkx >= 3.0); fall back to greedy modularity
try:
    communities = list(nx.community.louvain_communities(G_undirected_w, weight='weight', seed=42))
    method_used = 'Louvain'
except AttributeError:
    communities = list(nx.community.greedy_modularity_communities(G_undirected_w, weight='weight'))
    method_used = 'Greedy Modularity'

print(f'Community detection method : {method_used}')
print(f'Number of communities found: {len(communities)}')
for i, comm in enumerate(communities):
    print(f'  Community {i+1}: {sorted(comm)}')

# Build colour map
palette = plt.cm.Set1.colors
node_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_community[node] = i

node_colors_comm = [palette[node_community[n] % len(palette)] for n in G_undirected_w.nodes()]

edge_weights_u = [G_undirected_w[u][v].get('weight', 1) for u, v in G_undirected_w.edges()]
edge_widths_u  = [w / 3000 for w in edge_weights_u]

fig, ax = plt.subplots(figsize=(12, 8))
pos_comm = nx.spring_layout(G_undirected_w, seed=42)

nx.draw_networkx(
    G_undirected_w,
    pos=pos_comm,
    node_color=node_colors_comm,
    node_size=1200,
    font_size=8,
    font_color='white',
    edge_color='lightgray',
    width=edge_widths_u,
    ax=ax
)
patches = [
    mpatches.Patch(color=palette[i], label=f'Community {i+1}: {sorted(communities[i])}')
    for i in range(len(communities))
]
ax.legend(handles=patches, fontsize=8, loc='upper left')
ax.set_title(f'Community Detection ({method_used}) — Weighted Undirected Graph', fontsize=13)
plt.tight_layout()
plt.show()

## Cell 7 — Interpretation of communities

**Expected clusters and their meaning:**

The Louvain algorithm groups countries with strong mutual student-exchange flows. Based on the data, we expect to observe roughly:

- **Western European cluster** (France, Germany, Switzerland, Italy, United Kingdom): these countries exhibit very high bilateral flows, reflecting proximity, Erasmus+ programme participation, and shared academic traditions. Germany in particular acts as a hub, with enormous incoming flows from Turkey and Poland.
- **Eastern/Southern European cluster** (Greece, Turkey, Poland): these countries send large numbers of students westward but receive relatively few back — they act as net exporters of students. Their mutual flows are modest, but they share a structural role as peripheral senders.
- **East-Asian cluster** (Japan): Japan has lower absolute flow volumes with European neighbours and may form a singleton community or be attached to another cluster depending on the modularity resolution.
- **Nordic cluster** (Denmark): Denmark's volumes are small relative to the large Western European countries; it may cluster with the Western group or form its own small community.

**Interpretation of modularity-based clustering:**  
Community detection maximises the number of within-community edges relative to what would be expected by chance. High-weight edges (e.g., Germany ↔ United Kingdom, France ↔ Germany) pull nodes into the same community, while low-weight cross-links allow separation. The resulting clusters broadly correspond to **geographic proximity and shared mobility policy (Erasmus+)**, rather than pure directionality of flows.